# comms.io daily Colab run

This notebook keeps the SQLite checkpoint on Google Drive, runs a bounded crawl,
exports Parquet, validates it, and publishes the `data` branch.

Add `GITHUB_TOKEN` in the Colab secrets panel. The token needs `contents: write`
on `Elfiawesome/comms.io`.

The default settings are deliberately conservative. Rerun the notebook if the
session disconnects; the frontier resumes where it stopped.

In [ ]:
from pathlib import Path
import os

REPO_URL = "https://github.com/Elfiawesome/comms.io.git"
REPO_DIR = Path("/content/comms.io")

if REPO_DIR.exists():
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

In [ ]:
import sys
from pathlib import Path

venv = Path(".venv")
if not (venv / "bin" / "python").exists():
    !{sys.executable} -m venv .venv

PY = str(venv / "bin" / "python")
os.environ["PATH"] = str((venv / "bin").resolve()) + os.pathsep + os.environ.get("PATH", "")
!{PY} -m pip install --upgrade pip
!{PY} -m pip install -e .

# Optional: only needed if run_gallery_dl is enabled.
# !{PY} -m pip install gallery-dl

In [ ]:
from google.colab import drive, userdata

drive.mount("/content/drive")
checkpoint_dir = Path("/content/drive/MyDrive/comms.io")
gallery_dir = checkpoint_dir / "gallery-dl"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
gallery_dir.mkdir(parents=True, exist_ok=True)

os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")
os.environ["COMMS_DATABASE"] = str(checkpoint_dir / "comms.sqlite3")
os.environ["COMMS_GALLERY_DL_DIR"] = str(gallery_dir)

# Keep the export local; only the generated data branch is pushed.
os.environ["COMMS_EXPORT_DIR"] = str(REPO_DIR / "dist" / "data" / "v1")
print("checkpoint:", os.environ["COMMS_DATABASE"])
print("gallery-dl:", os.environ["COMMS_GALLERY_DL_DIR"])

## Optional: refresh the X/Twitter seed catalog

This uses the old gallery-dl workflow. It is optional because it depends on
your X cookies and gallery-dl continuing to work. Put `cookies.txt` in the
repository root or Drive if required by gallery-dl.

The crawler also imports any JSON already present in the gallery-dl directory.

In [ ]:
# Uncomment to run gallery-dl before the crawl.
# !{PY} -m comms crawl --run-gallery-dl --limit 0 --publish

In [ ]:
# Normal daily run: discover a little, refresh stale artists, then publish.
!{PY} -m comms crawl --limit 0 --publish

In [ ]:
!{PY} -m comms status
!{PY} -m comms validate

## Tuning

Edit `config/comms.toml` in the repo (or pass CLI options) to change:

- `run.max_requests` and `run.max_runtime_minutes`
- `run.max_new_artists` and `run.max_refresh_artists`
- `network.requests_per_second`
- `sources.skeb.discover_pages`

If repeated 429s occur, lower the rate and concurrency first. Do not disable
the adaptive limiter on shared IPs.